# Create Georgia CTSA Awards (Georgia Clinical and Translational Science Alliance pilot grants)

**What this is.** The Georgia CTSA (Emory University, Morehouse School of Medicine,
Georgia Institute of Technology, University of Georgia) is an NIH Clinical and
Translational Science Award hub. **Parent NIH grant: NCATS UL1TR002378** (Georgia CTSA,
2017-); predecessor hub ACTSI (Atlanta Clinical and Translational Science Institute,
UL1TR000454 / UL1RR025008). Its pilot grants are sub-awards paid out of that UL1 grant;
this notebook ingests them under the Georgia CTSA funder id, not under NIH.

**Source:** https://georgiactsa.org/funding/pilot-grant-recipients/index.html, one static
page per pilot-grant round (grants-<n>.html, rounds #34-#148), each listing project
title, amount and investigators (PI, Co-PI, Co-I, with position and institution).

**Scope filter (stated).** The same Emory pilot-grant office publishes rounds for many
other sponsors (Emory School of Medicine I3, Healthcare Innovation Program, Emory
University Research Committee, Regenerative Medicine, CFAR, ADRC, Morningside Center,
Gordon GCRC, Georgia Diabetes Translation Research Center, ...). Only rounds whose page
heading names the Georgia CTSA or its predecessor ACTSI are ingested: Georgia CTSA
(pilot translational & clinical studies), Georgia CTSA BERD, Informatics, REM Program,
Community Engagement, MSM/ACTSI Pilot and the co-funded "Georgia CTSA and James J Hills
and Wanda S Hills" round. Rounds #133 and #139 (Georgia CTSA, 9 and 10 awards) publish
only an award count, no list, so they are missing. 27 rounds, **114 awards,
2014-2026**; 100% title / amount / PI, 97% PI institution.

**`funder_award_id` (section 2.1.1):** Georgia CTSA publishes no pilot award numbers.
Citing works credit the hub with its NIH UL1/KL2/TL1 numbers (the ~299 existing
F4320316438 award stubs are UL1TR002378-style ids, many of them other hubs' UL1s
misattributed to this funder), which identify the parent NIH grant, not a pilot award.
Synthetic key: `GACTSA-R<round>-<nn>` (round number + 1-based position on the round
page). It collapses onto 0 existing stubs.

**Amounts** are the per-award USD figures printed on the round pages (direct pilot
funding). Dates: the round's announcement month is used as `start_date`; no end dates.
Mentors, consultants and advisers listed on a few awards are kept in the raw
`co_investigators` JSON but not emitted as investigators.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320316438`
- display_name: Georgia Clinical and Translational Science Alliance
- ror_id / doi: from `openalex.funders.funders` (ror 034hw7m59, doi 10.13039/100016220)

**Prerequisites:** run `scripts/local/georgia_ctsa_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/georgia_ctsa/georgia_ctsa_projects.parquet` (section 1.4 shrink guard).

**Priority:** `515` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.georgia_ctsa_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/georgia_ctsa/georgia_ctsa_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_award_ids FROM openalex.awards.georgia_ctsa_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.georgia_ctsa_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.georgia_ctsa_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320316438 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320316438;

## Step 2: Create Georgia CTSA Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.georgia_ctsa_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320316438  -- Georgia Clinical and Translational Science Alliance
),

g AS (
    SELECT
        r.*,
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE WHEN NULLIF(TRIM(r.pi_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(r.pi_given_name), ''),
            'family_name', NULLIF(TRIM(r.pi_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', CASE WHEN NULLIF(TRIM(r.pi_institution), '') IS NOT NULL THEN named_struct(
                'name', NULLIF(TRIM(r.pi_institution), ''),
                'country', 'US',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            ) END
        ) END AS lead,
        filter(
            transform(
                COALESCE(from_json(r.co_investigators,
                    'ARRAY<STRUCT<role:STRING, name:STRING, given_name:STRING, family_name:STRING, institution:STRING>>'),
                    array()),
                p -> CASE
                    WHEN NULLIF(TRIM(p.family_name), '') IS NOT NULL
                     AND (p.role IS NULL OR p.role RLIKE '(?i)^(co-? ?)?(pi|i|investigators?)$')
                    THEN named_struct(
                        'given_name', NULLIF(TRIM(p.given_name), ''),
                        'family_name', NULLIF(TRIM(p.family_name), ''),
                        'orcid', CAST(NULL AS STRING),
                        'role_start', CAST(NULL AS DATE),
                        'affiliation', CASE WHEN NULLIF(TRIM(p.institution), '') IS NOT NULL THEN named_struct(
                            'name', NULLIF(TRIM(p.institution), ''),
                            'country', 'US',
                            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                        ) END
                    ) END
            ),
            x -> x IS NOT NULL
        ) AS co
    FROM openalex.awards.georgia_ctsa_raw r
    WHERE r.funder_award_id IS NOT NULL AND TRIM(r.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('Georgia CTSA Pilot Grants: ', g.round_program) as funder_scheme,
    'georgia_ctsa_pilot_grants' as provenance,
    TRY_TO_DATE(g.round_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(TRY_TO_DATE(g.round_date, 'yyyy-MM-dd')) as start_year,
    CAST(NULL AS INT) as end_year,
    g.lead as lead_investigator,
    try_element_at(g.co, 1) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN concat(array(g.lead), g.co)
         WHEN size(g.co) > 0 THEN g.co END as investigators,
    g.round_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (section 2.1.1): every funder_award_id is GACTSA-R<round>-<nn>.
SELECT assert_true(
    COUNT_IF(NOT funder_award_id RLIKE '^GACTSA-R[0-9]+-[0-9]{2}$') = 0
    AND COUNT(*) = COUNT(DISTINCT id),
    'georgia_ctsa_awards: unexpected funder_award_id shape or duplicate ids'
) AS shape_ok
FROM openalex.awards.georgia_ctsa_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'georgia_ctsa_pilot_grants' AND priority = 515;

-- Insert into openalex_awards_raw with priority.
-- Priority 515: direct-from-funder ingest of Georgia CTSA pilot grant rounds.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    515 as priority
FROM openalex.awards.georgia_ctsa_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.georgia_ctsa_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, size(investigators) as n_inv
FROM openalex.awards.georgia_ctsa_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.georgia_ctsa_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.georgia_ctsa_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI names and titles.
SELECT lead_investigator.given_name, lead_investigator.family_name, COUNT(*) AS n
FROM openalex.awards.georgia_ctsa_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.georgia_ctsa_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.georgia_ctsa_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'georgia_ctsa_pilot_grants'
GROUP BY provenance, priority;